In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

### **DATASET EXPLORATION**

In [4]:
df=pd.read_csv('animal_disease_dataset.csv')
df.sample(5)

,Animal,Age,Temperature,Symptom 1,Symptom 2,Symptom 3,Disease
12402,buffalo,2,101.8,shortness of breath,fatigue,swelling in abdomen,anthrax
28935,sheep,5,100.4,loss of appetite,depression,painless lumps,pneumonia
14140,goat,1,102.7,loss of appetite,painless lumps,depression,lumpy virus
16477,buffalo,14,100.8,swelling in abdomen,fatigue,sweats,anthrax
2417,sheep,9,101.6,fatigue,swelling in extremities,shortness of breath,anthrax


In [5]:
df['Animal']=df['Animal'].astype(str).str.strip().str.lower()
cow_df=df[df['Animal']=='cow'].copy()

In [6]:
cow_df

,Animal,Age,Temperature,Symptom 1,Symptom 2,Symptom 3,Disease
0,cow,3,103.1,depression,painless lumps,loss of appetite,pneumonia
3,cow,14,100.3,loss of appetite,swelling in limb,crackling sound,blackleg
11,cow,11,103.9,depression,painless lumps,loss of appetite,lumpy virus
19,cow,14,102.7,shortness of breath,sweats,chills,anthrax
20,cow,1,103.7,depression,loss of appetite,painless lumps,lumpy virus
...,...,...,...,...,...,...,...
43759,cow,14,103.2,sores on gums,loss of appetite,lameness,foot and mouth
43760,cow,4,100.2,loss of appetite,painless lumps,depression,lumpy virus
43768,cow,8,102.4,depression,painless lumps,loss of appetite,lumpy virus
43770,cow,4,103.8,painless lumps,loss of appetite,depression,pneumonia


In [7]:
cow_df['Disease']=cow_df['Disease'].astype(str).str.strip().str.lower()

In [8]:
cow_df=cow_df.drop(columns=['Animal'])
cow_df.sample(5)

,Age,Temperature,Symptom 1,Symptom 2,Symptom 3,Disease
2689,5,101.3,sweats,fatigue,shortness of breath,anthrax
40272,3,101.3,crackling sound,difficulty walking,loss of appetite,blackleg
6831,1,100.1,blisters on tongue,loss of appetite,difficulty walking,foot and mouth
9172,10,100.4,loss of appetite,lameness,sores on tongue,foot and mouth
24986,6,104.4,lameness,loss of appetite,blisters on tongue,foot and mouth


In [9]:
X=cow_df.drop(columns=['Disease'])
y=cow_df['Disease']

In [10]:
print("Cow records:", len(cow_df))
print("Features:", X.columns.tolist())
print("\nDisease distribution:")
print(y.value_counts())

Cow records: 11254
Features: ['Age', 'Temperature', 'Symptom 1', 'Symptom 2', 'Symptom 3']

Disease distribution:
Disease
blackleg          2494
anthrax           2458
foot and mouth    2433
pneumonia         1955
lumpy virus       1914
Name: count, dtype: int64


### **FEATURE ENGINEERING / TRAIN-TEST-SPLIT**

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import  OneHotEncoder,StandardScaler
from sklearn.pipeline import Pipeline

In [12]:
numeric_features=['Age','Temperature']
categorical_features=['Symptom 1', 'Symptom 2', 'Symptom 3']

for col in categorical_features:
  X[col]=X[col].astype(str).str.strip().str.lower()

In [13]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [14]:
preprocessor=ColumnTransformer([
    (
        'numeric',StandardScaler(),numeric_features
    ),
    (
        'categorical',OneHotEncoder(handle_unknown='ignore',sparse_output=False),categorical_features
    )

])

In [15]:
print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

Training records: 9003
Testing records: 2251
Numerical features: ['Age', 'Temperature']
Categorical features: ['Symptom 1', 'Symptom 2', 'Symptom 3']


### **CLASSIFICATION MODELS**

In [16]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier
)
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

In [17]:
models={
    'Logistic Regression':LogisticRegression(max_iter=2000),
    'KNN':KNeighborsClassifier(),
    'SVM':SVC(),
    'Decision Tree':DecisionTreeClassifier(random_state=42),
    'Random Forest':RandomForestClassifier(n_estimators=200,n_jobs=-1,random_state=42),
    'Extra Trees':ExtraTreesClassifier(n_estimators=200,n_jobs=-1,random_state=42),
    'Gradient Boosting':GradientBoostingClassifier(random_state=42),
    'AdaBoosting':AdaBoostClassifier(random_state=42),
    'Gaussian Naive Bayes':GaussianNB()
}

results=[]
pipelines={}

for name,model in models.items():
  pipeline=Pipeline([
      ('preprocessor',clone(preprocessor)),
      ('model',model)
  ])
  pipeline.fit(X_train,y_train)
  predictions=pipeline.predict(X_test)
  accuracy=accuracy_score(y_test,predictions)
  f1=f1_score(y_test,predictions,average='macro')
  results.append({
      'Model':name,
      'Accuracy':accuracy*100,
      'Macro F1':f1*100
  })
  pipelines[name]=pipeline
  print(f"{name}: "
        f"Accuracy={accuracy*100:.2f}, "
        f"Macro F1={f1*100:.2f}")

baseline_results = pd.DataFrame(results).sort_values(
    "Macro F1", ascending=False
)

display(baseline_results)

Logistic Regression: Accuracy=82.36, Macro F1=79.39
KNN: Accuracy=79.34, Macro F1=76.18
SVM: Accuracy=80.72, Macro F1=77.55
Decision Tree: Accuracy=77.12, Macro F1=73.29
Random Forest: Accuracy=77.48, Macro F1=73.80
Extra Trees: Accuracy=77.88, Macro F1=74.26
Gradient Boosting: Accuracy=80.32, Macro F1=77.11
AdaBoosting: Accuracy=21.59, Macro F1=7.86
Gaussian Naive Bayes: Accuracy=82.41, Macro F1=79.50


,Model,Accuracy,Macro F1
8,Gaussian Naive Bayes,82.407819,79.499843
0,Logistic Regression,82.363394,79.393738
2,SVM,80.719680,77.552924
6,Gradient Boosting,80.319858,77.105905
1,KNN,79.342514,76.180369
5,Extra Trees,77.876499,74.260816
4,Random Forest,77.476677,73.798056
3,Decision Tree,77.121279,73.294744
7,AdaBoosting,21.590404,7.857720


## **LOGISTIC REGRESSION - HYP. TUN.**

In [18]:
from sklearn.model_selection import GridSearchCV,StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [19]:
lr_pipeline=Pipeline([
    ('preprocessor',preprocessor),
    ('model',LogisticRegression(max_iter=3000))
])

param_grid={
    'model__C':[0.01,0.1,1,10,100],
    'model__class_weight':[None,'balanced'],
    'model__solver':['lbfgs']
}

cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

lr_search=GridSearchCV(estimator=lr_pipeline,
                       param_grid=param_grid,
                       scoring='f1_macro',
                       cv=cv,
                       n_jobs=-1,
                       verbose=1)

lr_search.fit(X_train,y_train)

print('Best parameters: ',lr_search.best_params_)
print(f"Best CV Macro F1: {lr_search.best_score_ * 100:.2f}%")

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best parameters:  {'model__C': 100, 'model__class_weight': 'balanced', 'model__solver': 'lbfgs'}
Best CV Macro F1: 80.19%


## **SVM - HYP. TUN.**

In [20]:
svm_pipeline=Pipeline([
    ('preprocessor',preprocessor),
    ('model',SVC())
])

svm_param_grid=[{
    'model__kernel':['rbf'],
    'model__C':[0.1,1,10,100],
    'model__gamma':['scale',0.01,0.1],
    'model__class_weight':[None,'balanced']
},{
    'model__kernel':['linear'],
    'model__C':[0.1,1,10,100],
    'model__class_weight':[None,'balanced']
}]

svm_search=GridSearchCV(estimator=svm_pipeline,
                       param_grid=svm_param_grid,
                        scoring='f1_macro',
                        cv=cv,
                        n_jobs=-1,
                        verbose=1)
svm_search.fit(X_train,y_train)

print("Best parameters:", svm_search.best_params_)
print(f"Best CV Macro F1: {svm_search.best_score_ * 100:.2f}%")

Fitting 5 folds for each of 32 candidates, totalling 160 fits
Best parameters: {'model__C': 0.1, 'model__class_weight': None, 'model__kernel': 'linear'}
Best CV Macro F1: 79.94%


### **GNB - HYP. TUN.**

In [21]:
gnb_pipeline=Pipeline([
    ('preprocessor',preprocessor),
    ('model',GaussianNB())
])
gnb_param_grid={
    'model__var_smoothing':np.logspace(-12,-6,7)
}
gnb_search=GridSearchCV(estimator=gnb_pipeline,
                       param_grid=gnb_param_grid,
                        scoring='f1_macro',
                        cv=cv,
                        n_jobs=-1,
                        verbose=1)
gnb_search.fit(X_train,y_train)

print("Best parameters:", gnb_search.best_params_)
print(f"Best CV Macro F1: {gnb_search.best_score_ * 100:.2f}%")

Fitting 5 folds for each of 7 candidates, totalling 35 fits
Best parameters: {'model__var_smoothing': np.float64(1e-12)}
Best CV Macro F1: 80.35%


### **GRADIENT BOOSTING - HYP. TUN.**

In [22]:
gb_pipeline=Pipeline([
    ('preprocessor',preprocessor),
    ('model',GradientBoostingClassifier(random_state=42))
])
gb_param_grid={
    'model__n_estimators':[100,200],
    'model__learning_rate':[0.05,0.1],
    'model__max_depth':[2,3],
    'model__subsample':[0.8,1.0]
}

gb_search=GridSearchCV(estimator=gb_pipeline,
                       param_grid=gb_param_grid,
                        scoring='f1_macro',cv=cv,
                       n_jobs=-1,
                       verbose=1)
gb_search.fit(X_train,y_train)

print("Best parameters:", gb_search.best_params_)
print(f"Best CV Macro F1: {gb_search.best_score_ * 100:.2f}%")

Fitting 5 folds for each of 16 candidates, totalling 80 fits
Best parameters: {'model__learning_rate': 0.1, 'model__max_depth': 2, 'model__n_estimators': 200, 'model__subsample': 0.8}
Best CV Macro F1: 78.84%


### **COMPARING MODELS**

In [26]:
tuned_results=pd.DataFrame([
    {
        'Model':'Logistic Regression',
        'CV Macro F1':lr_search.best_score_,
        'Best Estimator':lr_search.best_estimator_
    },
    {
        'Model':'SVM',
        'CV Macro F1':svm_search.best_score_,
        'Best Estimator':svm_search.best_estimator_
    },
    {
        'Model':'Gaussian Naive Bayes',
        'CV Macro F1':gnb_search.best_score_,
        'Best Estimator':gnb_search.best_estimator_
    },
    {
        'Model':'Gradient Boosting',
        'CV Macro F1':gb_search.best_score_,
        'Best Estimator':gb_search.best_estimator_
    }
])

tuned_results=tuned_results.sort_values('CV Macro F1',ascending=False).reset_index(drop=True)

display(
    tuned_results[["Model", "CV Macro F1"]]
    .assign(**{"CV Macro F1 (%)": lambda d: d["CV Macro F1"] * 100})
    [["Model", "CV Macro F1 (%)"]]
)


best_model_name = tuned_results.loc[0, "Model"]
best_pipeline = tuned_results.loc[0, "Best Estimator"]

print("\nCurrent best model:", best_model_name)
print("\nCurrent best model pipeline:", best_pipeline)

,Model,CV Macro F1 (%)
0,Gaussian Naive Bayes,80.345228
1,Logistic Regression,80.188964
2,SVM,79.936736
3,Gradient Boosting,78.844733



Current best model: Gaussian Naive Bayes

Current best model pipeline: Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric', StandardScaler(),
                                                  ['Age', 'Temperature']),
                                                 ('categorical',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['Symptom 1', 'Symptom 2',
                                                   'Symptom 3'])])),
                ('model', GaussianNB(var_smoothing=np.float64(1e-12)))])


### **EVALUATION OF THE BEST MODEL**

In [27]:
from sklearn.metrics import classification_report,confusion_matrix

y_pred=best_pipeline.predict(X_test)

print('Model:',best_model_name)
print(f'Test Accuracy: {accuracy_score(y_test,y_pred)*100:.2f}%')
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))

Model: Gaussian Naive Bayes
Test Accuracy: 82.41%
                precision    recall  f1-score   support

       anthrax       1.00      1.00      1.00       492
      blackleg       1.00      1.00      1.00       499
foot and mouth       1.00      1.00      1.00       486
   lumpy virus       0.48      0.54      0.51       383
     pneumonia       0.49      0.44      0.47       391

      accuracy                           0.82      2251
     macro avg       0.80      0.80      0.79      2251
  weighted avg       0.82      0.82      0.82      2251

[[492   0   0   0   0]
 [  0 499   0   0   0]
 [  0   0 486   0   0]
 [  0   0   0 205 178]
 [  0   0   0 218 173]]


### **CHECKING AMBIGUITY**

In [34]:
# for disease in ["lumpy virus", "pneumonia"]:
#     print(f"\n--- {disease.upper()} ---")

#     display(
#         cow_df[cow_df["Disease"] == disease][
#             ["Age", "Temperature", "Symptom 1",
#              "Symptom 2", "Symptom 3"]
#         ].describe(include="all").T
#     )

In [35]:
# symptom_cols = ["Symptom 1", "Symptom 2", "Symptom 3"]

# pattern_counts = (
#     cow_df.groupby(symptom_cols)["Disease"]
#     .nunique()
#     .reset_index(name="number_of_diseases")
# )

# overlapping_patterns = pattern_counts[
#     pattern_counts["number_of_diseases"] > 1
# ]

# print("Symptom patterns appearing in multiple diseases:",
#       len(overlapping_patterns))

# display(overlapping_patterns.head(20))

In [36]:
# symptom_cols = ["Symptom 1", "Symptom 2", "Symptom 3"]

# # Count records for every symptom pattern and disease
# pattern_disease_counts = (
#     cow_df.groupby(symptom_cols + ["Disease"])
#     .size()
#     .unstack(fill_value=0)
# )

# # Patterns associated with more than one disease
# ambiguous_patterns = pattern_disease_counts[
#     (pattern_disease_counts > 0).sum(axis=1) > 1
# ]

# print("Ambiguous symptom patterns:", len(ambiguous_patterns))
# print("Records with ambiguous patterns:",
#       ambiguous_patterns.sum(axis=1).sum())

# display(ambiguous_patterns)

In [37]:
# ambiguous_df = cow_df.merge(
#     ambiguous_patterns.reset_index()[
#         ["Symptom 1", "Symptom 2", "Symptom 3"]
#     ],
#     on=["Symptom 1", "Symptom 2", "Symptom 3"],
#     how="inner"
# )

# print("Records:", len(ambiguous_df))

# display(
#     ambiguous_df.groupby("Disease")[
#         ["Age", "Temperature"]
#     ].agg(["mean", "std", "min", "max"]).round(2)
# )

In [38]:
# feature_cols = [
#     "Age",
#     "Temperature",
#     "Symptom 1",
#     "Symptom 2",
#     "Symptom 3"
# ]

# # Count distinct disease labels for each exact feature combination
# label_counts = (
#     cow_df.groupby(feature_cols)["Disease"]
#     .nunique()
# )

# conflicting_features = label_counts[label_counts > 1]

# print("Conflicting feature combinations:", len(conflicting_features))
# print(
#     "Records in conflicting combinations:",
#     cow_df.set_index(feature_cols)
#     .loc[conflicting_features.index]
#     .shape[0]
# )

# display(conflicting_features.head(10))

### **SAVE MODEL**

In [40]:
import joblib
import pickle
from google.colab import files
joblib.dump(best_pipeline,'cow_tabular_model.joblib')
files.download('cow_tabular_model.joblib')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>